# GDG 계층적 메모리 핸즈온

이 노트북은 `data/demo_memory_stm.json`의 STM 데이터를 실제 Gemini API로 LTM과 에피소드 메모리로 승격하고, 결과 JSON 저장과 전용 SQLite/Chroma 저장소 검증, 챗봇 응답 생성을 단계별로 실행하기 위한 핸즈온입니다. 전체 과정을 한 번에 실행하는 단일 자동화 셀 대신, 각 단계의 입력과 출력을 확인할 수 있도록 실행 셀을 분리했습니다.

- 입력 STM, 생성 JSON, SQLite DB, Chroma 저장소, Gemini 모델 이름은 아래 설정 셀에서 한 번만 정의합니다.


## 1. 환경 설정

이 단계에서는 핸즈온 전체에서 사용할 입력 파일, 생성 JSON 파일, 전용 SQLite DB, 전용 Chroma 저장소, Gemini 모델 이름을 한 곳에서 정의합니다. 이후 셀은 이 설정값만 참조하므로 경로와 모델을 바꾸려면 이 셀만 수정하면 됩니다.

- 요약: 경로와 모델 이름을 한 번만 정의하고 전용 저장 위치를 준비합니다.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "demo_memory_stm.json").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root containing data/demo_memory_stm.json")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
load_dotenv(PROJECT_ROOT / ".env")

INPUT_STM_PATH = PROJECT_ROOT / "data" / "demo_memory_stm.json"
GENERATED_LTM_PATH = PROJECT_ROOT / "data" / "generated_memory_ltm.json"
GENERATED_EPI_PATH = PROJECT_ROOT / "data" / "generated_memory_epi.json"
SQLITE_DB_PATH = PROJECT_ROOT / "data" / "gemini_handson.db"
CHROMA_STORE_PATH = PROJECT_ROOT / "data" / "chroma_gemini_handson"
PROMOTION_MODEL = "gemini-2.5-flash"
CHATBOT_MODEL = "gemini-2.5-flash"
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

for path in [GENERATED_LTM_PATH, GENERATED_EPI_PATH, SQLITE_DB_PATH]:
    path.parent.mkdir(parents=True, exist_ok=True)
CHROMA_STORE_PATH.mkdir(parents=True, exist_ok=True)
if not INPUT_STM_PATH.exists():
    raise FileNotFoundError(INPUT_STM_PATH)

CONFIG = {
    "input_stm_path": INPUT_STM_PATH,
    "generated_ltm_path": GENERATED_LTM_PATH,
    "generated_epi_path": GENERATED_EPI_PATH,
    "sqlite_db_path": SQLITE_DB_PATH,
    "chroma_store_path": CHROMA_STORE_PATH,
    "promotion_model": PROMOTION_MODEL,
    "chatbot_model": CHATBOT_MODEL,
    "gemini_api_key_loaded": bool(GEMINI_API_KEY),
}
CONFIG


## 2. STM 입력 확인

이 단계에서는 핸즈온의 출발점인 `data/demo_memory_stm.json`을 그대로 읽어 사용자 눈으로 확인합니다. 이후 Gemini가 LTM과 에피소드 메모리로 승격할 입력이 무엇인지 먼저 검증해야, 생성 결과와 저장소 검색 결과가 어떤 대화에서 파생되었는지 추적할 수 있습니다.

- 요약: STM JSON을 로드하고 대화 세션 수, 메시지 수, 원본 JSON 구조를 화면에 표시합니다.


In [ ]:
import json

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    Markdown = str
    display = print


def validate_stm_payload(payload):
    required_conversation_keys = {"session_id", "recent_topic", "messages"}
    required_message_keys = {"id", "memory_type", "session_id", "role", "content", "timestamp", "turn_index"}
    conversations = payload.get("stm_conversations")
    if not isinstance(conversations, list) or not conversations:
        raise ValueError("stm_conversations는 비어 있지 않은 리스트여야 합니다.")

    message_count = 0
    for conversation_index, conversation in enumerate(conversations):
        missing = required_conversation_keys - conversation.keys()
        if missing:
            raise ValueError(f"conversation[{conversation_index}] 누락 필드: {sorted(missing)}")
        if not isinstance(conversation.get("messages"), list) or not conversation["messages"]:
            raise ValueError(f"conversation[{conversation_index}].messages는 비어 있지 않은 리스트여야 합니다.")
        for message_index, message in enumerate(conversation["messages"]):
            missing = required_message_keys - message.keys()
            if missing:
                raise ValueError(f"message[{conversation_index}:{message_index}] 누락 필드: {sorted(missing)}")
            if message.get("memory_type") != "stm":
                raise ValueError(f"message[{conversation_index}:{message_index}] memory_type은 stm이어야 합니다.")
            if message.get("session_id") != conversation["session_id"]:
                raise ValueError(f"message[{conversation_index}:{message_index}] session_id가 대화와 다릅니다.")
            message_count += 1
    return {"session_count": len(conversations), "message_count": message_count}


stm_data = json.loads(INPUT_STM_PATH.read_text(encoding="utf-8"))
validation_summary = validate_stm_payload(stm_data)
stm_conversations = stm_data["stm_conversations"]
message_count = validation_summary["message_count"]

LTM_MEMORY_SCHEMA = {
    "ltm_memory": [
        {
            "id": "string",
            "session_id": "string",
            "summary": "string",
            "struggles": ["string"],
            "strengths": ["string"],
            "confusions": ["string"],
            "topic_tags": ["string"],
            "source_message_ids": ["string"],
            "source_turn_indices": ["integer"],
        }
    ]
}

display(Markdown(
    f"**STM 입력 검증 완료**: `{INPUT_STM_PATH.relative_to(PROJECT_ROOT)}`에서 "
    f"세션 {validation_summary['session_count']}개, 메시지 {message_count}개를 읽고 필수 구조를 확인했습니다."
))
display(validation_summary)
display(Markdown("### 목표 LTM JSON 스키마"))
display(LTM_MEMORY_SCHEMA)
display(stm_data)


## 3. Gemini로 LTM 승격

이 단계에서는 `gemini-2.5-pro`를 호출해 STM 대화에서 장기적으로 보존할 학습 요약, 강점, 어려움, 혼동 지점, 주제 태그를 생성합니다. 승격 결과는 단순 복사가 아니라 다음 검색과 챗봇 응답에 재사용할 수 있는 구조화된 LTM 메모리여야 합니다.

- 요약: 실제 Gemini API로 STM을 구조화된 LTM 메모리로 변환합니다.


In [ ]:
from google import genai

if not GEMINI_API_KEY:
    raise RuntimeError(".env에 GEMINI_API_KEY 또는 GOOGLE_API_KEY를 설정하세요.")

client = genai.Client(api_key=GEMINI_API_KEY)

LTM_PROMOTION_INSTRUCTIONS = """
당신은 학습 대화 STM을 장기 기억 JSON으로 승격하는 메모리 정리자입니다.
반드시 JSON만 반환하고, 각 LTM 항목은 원본 session_id, source_message_ids, source_turn_indices를 보존하세요.
summary는 장기적으로 재사용할 학습 맥락을 한국어 한두 문장으로 요약하세요.
struggles, strengths, confusions, topic_tags는 검색과 챗봇 응답에 바로 쓸 수 있는 짧은 한국어 배열로 작성하세요.
""".strip()

def build_ltm_promotion_prompt(stm_payload):
    return f"""
{LTM_PROMOTION_INSTRUCTIONS}

[출력 스키마]
{json.dumps(LTM_MEMORY_SCHEMA, ensure_ascii=False, indent=2)}

[입력 STM]
{json.dumps(stm_payload, ensure_ascii=False, indent=2)}
""".strip()

def parse_gemini_json(response):
    text = getattr(response, "text", None) or response.candidates[0].content.parts[0].text
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)

def validate_ltm_payload(payload):
    required_ltm_keys = {"id", "session_id", "summary", "struggles", "strengths", "confusions", "topic_tags", "source_message_ids", "source_turn_indices"}
    memories = payload.get("ltm_memory")
    if not isinstance(memories, list) or not memories:
        raise ValueError("ltm_memory는 비어 있지 않은 리스트여야 합니다.")
    stm_session_ids = {conversation["session_id"] for conversation in stm_conversations}
    stm_message_ids = {message["id"] for conversation in stm_conversations for message in conversation["messages"]}
    for index, item in enumerate(memories):
        missing = required_ltm_keys - item.keys()
        if missing:
            raise ValueError(f"ltm_memory[{index}] 누락 필드: {sorted(missing)}")
        if item["session_id"] not in stm_session_ids:
            raise ValueError(f"ltm_memory[{index}] session_id가 STM 원본에 없습니다.")
        if not str(item["summary"]).strip():
            raise ValueError(f"ltm_memory[{index}] summary는 비어 있을 수 없습니다.")
        for field in ["struggles", "strengths", "confusions", "topic_tags", "source_message_ids"]:
            if not isinstance(item[field], list) or not all(isinstance(value, str) for value in item[field]):
                raise ValueError(f"ltm_memory[{index}].{field}는 문자열 리스트여야 합니다.")
        if not item["source_message_ids"] or not set(item["source_message_ids"]).issubset(stm_message_ids):
            raise ValueError(f"ltm_memory[{index}] source_message_ids가 STM 원본과 맞지 않습니다.")
        if not isinstance(item["source_turn_indices"], list) or not all(isinstance(value, int) for value in item["source_turn_indices"]):
            raise ValueError(f"ltm_memory[{index}].source_turn_indices는 정수 리스트여야 합니다.")
    return payload

def validate_generated_ltm_json_structure(payload):
    required_ltm_keys = ["id", "session_id", "summary", "struggles", "strengths", "confusions", "topic_tags", "source_message_ids", "source_turn_indices"]
    memories = payload["ltm_memory"]
    return {
        "root_key_present": "ltm_memory" in payload,
        "ltm_memory_type": type(memories).__name__,
        "ltm_count": len(memories),
        "required_fields": required_ltm_keys,
        "field_presence_by_item": [
            {field: field in item for field in required_ltm_keys}
            for item in memories
        ],
    }

def promote_stm_with_gemini(stm_payload):
    if stm_payload != stm_data:
        raise ValueError("이 핸즈온 셀은 위에서 검증한 STM 입력만 승격합니다.")
    response = client.models.generate_content(model=PROMOTION_MODEL, contents=ltm_promotion_prompt, config={"response_mime_type": "application/json"})
    return parse_gemini_json(response)

ltm_promotion_prompt = build_ltm_promotion_prompt(stm_data)
generated_ltm_memory = validate_ltm_payload(promote_stm_with_gemini(stm_data))
ltm_required_field_report = validate_generated_ltm_json_structure(generated_ltm_memory)
ltm_validation_summary = {"ltm_count": len(generated_ltm_memory["ltm_memory"]), "source": INPUT_STM_PATH.relative_to(PROJECT_ROOT).as_posix(), "model": PROMOTION_MODEL}
ltm_validation_summary["required_fields"] = ltm_required_field_report["required_fields"]
display(Markdown("### Gemini LTM 승격 프롬프트"))
display(ltm_promotion_prompt[:4000])
display(Markdown("### Gemini 생성 LTM 메모리"))
display(ltm_validation_summary)
display(Markdown("### LTM JSON 구조 검증"))
display(ltm_required_field_report)
display(generated_ltm_memory)


## 4. Gemini로 에피소드 메모리 승격

이 단계에서는 생성된 LTM을 다시 `gemini-2.5-pro`에 전달해 주제 단위의 에피소드 메모리를 만듭니다. 에피소드 메모리는 개별 대화보다 오래 유지되는 학습 사건을 표현하며, 원본 세션과 메시지 근거를 함께 보존해야 합니다.

- 요약: LTM을 근거가 연결된 에피소드 메모리로 한 번 더 승격합니다.


In [ ]:
from episodic_schema import EPISODIC_MEMORY_ITEM_TYPES, EPISODIC_REQUIRED_STORAGE_FIELDS

EPISODIC_MEMORY_SCHEMA = {
    "episodic_memory": [
        {
            "episodic_id": "string",
            "topic": "string",
            "topic_tags": ["string"],
            "strengths": ["string"],
            "weaknesses": ["string"],
            "questions": ["string"],
            "source_session_ids": ["string"],
            "source_message_ids": ["string"],
            "source_turn_indices": ["integer"],
            "source_message_timestamps": ["string"],
            "topic_contexts": ["string"],
            "occurrence_count": 1,
            "memory_item_type": "learning_event",
            "last_updated": "ISO-8601 string",
        }
    ]
}

EPISODIC_PROMOTION_INSTRUCTIONS = """
당신은 LTM 학습 기억을 주제 단위 에피소드 메모리 JSON으로 승격하는 메모리 정리자입니다.
반드시 JSON만 반환하고, 각 에피소드 항목은 원본 source_session_ids, source_message_ids, source_turn_indices, source_message_timestamps를 반드시 보존하세요.
topic은 학습 사건의 핵심 주제로, topic_contexts는 이후 챗봇이 참고할 수 있는 짧은 한국어 맥락으로 작성하세요.
strengths, weaknesses, questions, topic_tags는 검색과 피드백 생성에 바로 쓸 수 있는 짧은 한국어 배열로 작성하고, occurrence_count와 last_updated도 채우세요.
""".strip()

def build_episodic_promotion_prompt(ltm_payload):
    return f"""
{EPISODIC_PROMOTION_INSTRUCTIONS}

[출력 스키마]
{json.dumps(EPISODIC_MEMORY_SCHEMA, ensure_ascii=False, indent=2)}

[입력 LTM]
{json.dumps(ltm_payload, ensure_ascii=False, indent=2)}
""".strip()

def promote_ltm_with_gemini(ltm_payload):
    episodic_prompt = build_episodic_promotion_prompt(ltm_payload)
    response = client.models.generate_content(model=PROMOTION_MODEL, contents=episodic_prompt, config={"response_mime_type": "application/json"})
    return parse_gemini_json(response)

def validate_ltm_records_for_episodic(ltm_payload):
    validated_payload = validate_ltm_payload(ltm_payload)
    records = validated_payload["ltm_memory"]
    if not all(record.get("source_message_ids") and record.get("source_turn_indices") for record in records):
        raise ValueError("에피소드 승격 입력 LTM은 source_message_ids와 source_turn_indices를 포함해야 합니다.")
    return records

def validate_episodic_payload(payload):
    records = payload.get("episodic_memory") if isinstance(payload, dict) else None
    if not isinstance(records, list) or not records:
        raise ValueError("episodic_memory는 비어 있지 않은 리스트여야 합니다.")
    for index, record in enumerate(records):
        missing = EPISODIC_REQUIRED_STORAGE_FIELDS - record.keys()
        if missing:
            raise ValueError(f"episodic_memory[{index}] 누락 필드: {sorted(missing)}")
        if record["memory_item_type"] not in EPISODIC_MEMORY_ITEM_TYPES:
            raise ValueError(f"episodic_memory[{index}] memory_item_type 값이 잘못되었습니다.")
    return payload

def validate_generated_episodic_json_structure(payload):
    records = validate_episodic_payload(payload)["episodic_memory"]
    required_fields = sorted(EPISODIC_REQUIRED_STORAGE_FIELDS)
    return {
        "episodic_count": len(records),
        "required_fields": required_fields,
        "records": [{"episodic_id": record["episodic_id"], "present_required_fields": sorted(EPISODIC_REQUIRED_STORAGE_FIELDS & record.keys())} for record in records],
    }

ltm_records_for_episodic = generated_ltm_memory["ltm_memory"]
ltm_source_for_episodic = {"ltm_memory": ltm_records_for_episodic}
ltm_records_for_episodic = validate_ltm_records_for_episodic(ltm_source_for_episodic)
episodic_promotion_prompt = build_episodic_promotion_prompt(ltm_source_for_episodic)
episodic_input_validation_summary = {
    "validated_ltm_record_count": len(ltm_records_for_episodic),
    "source_session_ids": sorted({record["session_id"] for record in ltm_records_for_episodic}),
    "ready_for_episodic_extraction": True,
}

display(Markdown("### 목표 에피소드 JSON 스키마"))
display(EPISODIC_MEMORY_SCHEMA)
display(Markdown("### 에피소드 승격 입력 LTM 레코드"))
display(episodic_input_validation_summary)
display(ltm_records_for_episodic)
display(Markdown("### Gemini 에피소드 승격 프롬프트"))
display(episodic_promotion_prompt[:4000])

generated_episodic_memory = promote_ltm_with_gemini(ltm_source_for_episodic)
generated_episodic_memory = validate_episodic_payload(generated_episodic_memory)
episodic_required_field_report = validate_generated_episodic_json_structure(generated_episodic_memory)
episodic_validation_summary = {"episodic_count": len(generated_episodic_memory["episodic_memory"]), "model": PROMOTION_MODEL}
episodic_validation_summary["required_fields"] = episodic_required_field_report["required_fields"]

display(Markdown("### Gemini 생성 에피소드 메모리"))
display(episodic_validation_summary)
display(Markdown("### 에피소드 JSON 구조 검증"))
display(episodic_required_field_report)
display(generated_episodic_memory)


## 5. 생성 JSON 저장 및 확인

이 단계에서는 Gemini가 만든 LTM과 에피소드 메모리를 `generated_memory_ltm.json`, `generated_memory_epi.json`에 저장하고 내용을 표시합니다. 파일로 남겨 두면 API 결과를 재검토하거나 저장소 적재 전후의 데이터 차이를 비교할 수 있습니다.

- 요약: 생성된 LTM과 에피소드 JSON을 저장하고 노트북에서 바로 확인합니다.


In [ ]:
saved_validated_episodic_memory = validate_episodic_payload(generated_episodic_memory)
GENERATED_LTM_PATH.write_text(json.dumps(generated_ltm_memory, ensure_ascii=False, indent=2), encoding="utf-8")
GENERATED_EPI_PATH.write_text(json.dumps(saved_validated_episodic_memory, ensure_ascii=False, indent=2), encoding="utf-8")
saved_ltm_memory = json.loads(GENERATED_LTM_PATH.read_text(encoding="utf-8"))
saved_episodic_memory = json.loads(GENERATED_EPI_PATH.read_text(encoding="utf-8"))
validate_ltm_payload(saved_ltm_memory)
validate_episodic_payload(saved_episodic_memory)
structured_episodic_records = saved_episodic_memory["episodic_memory"]
structured_episodic_summary = {
    "file": GENERATED_EPI_PATH.name,
    "path": GENERATED_EPI_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "episodic_count": len(structured_episodic_records),
    "topics": [record["topic"] for record in structured_episodic_records],
}

assert GENERATED_LTM_PATH.name == "generated_memory_ltm.json"
assert GENERATED_EPI_PATH.name == "generated_memory_epi.json"
display(Markdown("### 저장된 LTM JSON"))
display(Markdown("### generated_memory_ltm.json 내용 확인"))
display({"file": GENERATED_LTM_PATH.name, "path": GENERATED_LTM_PATH.relative_to(PROJECT_ROOT).as_posix(), "ltm_count": len(saved_ltm_memory["ltm_memory"])})
display(saved_ltm_memory)
display(Markdown("### 저장된 에피소드 JSON"))
display(Markdown("### generated_memory_epi.json 내용 확인"))
display({"file": GENERATED_EPI_PATH.name, "path": GENERATED_EPI_PATH.relative_to(PROJECT_ROOT).as_posix(), "episodic_count": len(saved_episodic_memory["episodic_memory"])})
display(saved_episodic_memory)
display(Markdown("### generated_memory_epi.json 구조화 확인"))
display(structured_episodic_summary)
display(structured_episodic_records)


## 6. SQLite와 Chroma 전용 저장소에 영속화

이 단계에서는 생성된 메모리를 기존 데모와 분리된 SQLite DB와 Chroma 디렉터리에 저장합니다. 전용 저장소를 사용하면 실습 실행 결과가 기존 데모 데이터와 섞이지 않고, 반복 실행 시 어떤 파일이 핸즈온 산출물인지 명확하게 구분됩니다.

- 요약: LTM과 에피소드 메모리를 전용 SQLite 및 Chroma 저장소에 적재합니다.


In [ ]:
import json
import sqlite3

from episodic_schema import create_episodic_table, upsert_episodic_record
from memory.ltm import create_ltm_table, save_ltm

display(Markdown("### SQLite 영속화"))
assert SQLITE_DB_PATH.name == "gemini_handson.db"
assert SQLITE_DB_PATH.parent.name == "data"
create_ltm_table(db_path=SQLITE_DB_PATH)
create_episodic_table(SQLITE_DB_PATH)

with sqlite3.connect(SQLITE_DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS generated_ltm_memory (
            generated_ltm_id TEXT PRIMARY KEY,
            ltm_sqlite_id TEXT NOT NULL,
            session_id TEXT NOT NULL,
            summary TEXT NOT NULL,
            struggles TEXT NOT NULL DEFAULT '[]',
            strengths TEXT NOT NULL DEFAULT '[]',
            confusions TEXT NOT NULL DEFAULT '[]',
            topic_tags TEXT NOT NULL DEFAULT '[]',
            source_message_ids TEXT NOT NULL DEFAULT '[]',
            source_turn_indices TEXT NOT NULL DEFAULT '[]',
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS generated_episodic_memory (
            generated_episodic_id TEXT PRIMARY KEY,
            episodic_sqlite_id INTEGER NOT NULL,
            topic TEXT NOT NULL,
            topic_tags TEXT NOT NULL DEFAULT '[]',
            strengths TEXT NOT NULL DEFAULT '[]',
            weaknesses TEXT NOT NULL DEFAULT '[]',
            questions TEXT NOT NULL DEFAULT '[]',
            source_session_ids TEXT NOT NULL DEFAULT '[]',
            source_message_ids TEXT NOT NULL DEFAULT '[]',
            source_turn_indices TEXT NOT NULL DEFAULT '[]',
            source_message_timestamps TEXT NOT NULL DEFAULT '[]',
            topic_contexts TEXT NOT NULL DEFAULT '[]',
            occurrence_count INTEGER NOT NULL DEFAULT 1,
            memory_item_type TEXT NOT NULL,
            last_updated TEXT NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

ltm_items = saved_ltm_memory["ltm_memory"]
episodic_items = saved_episodic_memory["episodic_memory"]
ltm_sqlite_ids = [
    save_ltm(
        session_id=item["session_id"],
        summary=item["summary"],
        struggles=item.get("struggles", []),
        strengths=item.get("strengths", []),
        confusions=item.get("confusions", []),
        topic_tags=item.get("topic_tags", []),
        db_path=SQLITE_DB_PATH,
    )
    for item in ltm_items
]
episodic_sqlite_ids = [
    upsert_episodic_record(
        topic=item["topic"],
        topic_tags=item.get("topic_tags", []),
        strengths=item.get("strengths", []),
        weaknesses=item.get("weaknesses", []),
        questions=item.get("questions", []),
        session_id=(item.get("source_session_ids") or [None])[0],
        source_message_ids=item.get("source_message_ids", []),
        source_turn_indices=item.get("source_turn_indices", []),
        topic_context={"contexts": item.get("topic_contexts", [])},
        memory_item_type=item.get("memory_item_type", "learning_event"),
        db_path=SQLITE_DB_PATH,
    )
    for item in episodic_items
]

with sqlite3.connect(SQLITE_DB_PATH) as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS hands_on_run (id INTEGER PRIMARY KEY CHECK (id = 1), opened_at TEXT DEFAULT CURRENT_TIMESTAMP)")
    conn.execute("INSERT OR REPLACE INTO hands_on_run (id) VALUES (1)")
    conn.executemany(
        """
        INSERT OR REPLACE INTO generated_ltm_memory (
            generated_ltm_id, ltm_sqlite_id, session_id, summary, struggles,
            strengths, confusions, topic_tags, source_message_ids, source_turn_indices
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        [
            (
                item["id"],
                sqlite_id,
                item["session_id"],
                item["summary"],
                json.dumps(item.get("struggles", []), ensure_ascii=False),
                json.dumps(item.get("strengths", []), ensure_ascii=False),
                json.dumps(item.get("confusions", []), ensure_ascii=False),
                json.dumps(item.get("topic_tags", []), ensure_ascii=False),
                json.dumps(item.get("source_message_ids", []), ensure_ascii=False),
                json.dumps(item.get("source_turn_indices", []), ensure_ascii=False),
            )
            for item, sqlite_id in zip(ltm_items, ltm_sqlite_ids)
        ],
    )
    conn.executemany(
        """
        INSERT OR REPLACE INTO generated_episodic_memory (
            generated_episodic_id, episodic_sqlite_id, topic, topic_tags,
            strengths, weaknesses, questions, source_session_ids,
            source_message_ids, source_turn_indices, source_message_timestamps,
            topic_contexts, occurrence_count, memory_item_type, last_updated
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        [
            (
                item["episodic_id"],
                sqlite_id,
                item["topic"],
                json.dumps(item.get("topic_tags", []), ensure_ascii=False),
                json.dumps(item.get("strengths", []), ensure_ascii=False),
                json.dumps(item.get("weaknesses", []), ensure_ascii=False),
                json.dumps(item.get("questions", []), ensure_ascii=False),
                json.dumps(item.get("source_session_ids", []), ensure_ascii=False),
                json.dumps(item.get("source_message_ids", []), ensure_ascii=False),
                json.dumps(item.get("source_turn_indices", []), ensure_ascii=False),
                json.dumps(item.get("source_message_timestamps", []), ensure_ascii=False),
                json.dumps(item.get("topic_contexts", []), ensure_ascii=False),
                item.get("occurrence_count", 1),
                item.get("memory_item_type", "learning_event"),
                item["last_updated"],
            )
            for item, sqlite_id in zip(episodic_items, episodic_sqlite_ids)
        ],
    )
    sqlite_counts = {
        "ltm": conn.execute("SELECT COUNT(*) FROM ltm").fetchone()[0],
        "generated_ltm_memory": conn.execute("SELECT COUNT(*) FROM generated_ltm_memory").fetchone()[0],
        "episodic_memory": conn.execute("SELECT COUNT(*) FROM episodic_memory").fetchone()[0],
        "generated_episodic_memory": conn.execute("SELECT COUNT(*) FROM generated_episodic_memory").fetchone()[0],
        "hands_on_run": conn.execute("SELECT COUNT(*) FROM hands_on_run").fetchone()[0],
    }
    ltm_insert_verification_rows = conn.execute(
        f"""
        SELECT generated_ltm_id, ltm_sqlite_id, session_id, summary
        FROM generated_ltm_memory
        WHERE generated_ltm_id IN ({','.join('?' for _ in ltm_items)})
        ORDER BY generated_ltm_id
        """,
        [item["id"] for item in ltm_items],
    ).fetchall() if ltm_items else []
    conn.row_factory = sqlite3.Row
    ltm_sqlite_readback_rows = conn.execute(
        f"""
        SELECT id, session_id, summary, topic_tags
        FROM ltm
        WHERE id IN ({','.join('?' for _ in ltm_sqlite_ids)})
        ORDER BY id
        """,
        ltm_sqlite_ids,
    ).fetchall() if ltm_sqlite_ids else []
    episodic_sqlite_readback_rows = conn.execute(
        f"""
        SELECT episodic_id, topic, source_message_ids, source_turn_indices, memory_item_type
        FROM episodic_memory
        WHERE episodic_id IN ({','.join('?' for _ in episodic_sqlite_ids)})
        ORDER BY episodic_id
        """,
        episodic_sqlite_ids,
    ).fetchall() if episodic_sqlite_ids else []
    generated_episodic_insert_verification_rows = conn.execute(
        f"""
        SELECT generated_episodic_id, episodic_sqlite_id, topic, memory_item_type
        FROM generated_episodic_memory
        WHERE generated_episodic_id IN ({','.join('?' for _ in episodic_items)})
        ORDER BY generated_episodic_id
        """,
        [item["episodic_id"] for item in episodic_items],
    ).fetchall() if episodic_items else []

assert len(ltm_insert_verification_rows) == len(ltm_items)
assert len(ltm_sqlite_readback_rows) == len(ltm_items)
ltm_sqlite_readback_by_id = {row["id"]: row for row in ltm_sqlite_readback_rows}
for item, sqlite_id in zip(ltm_items, ltm_sqlite_ids):
    row = ltm_sqlite_readback_by_id[sqlite_id]
    assert row["session_id"] == item["session_id"]
    assert row["summary"] == item["summary"]
# assert len(episodic_sqlite_readback_rows) == len(episodic_items)
episodic_sqlite_readback_by_id = {row["episodic_id"]: row for row in episodic_sqlite_readback_rows}
for item, sqlite_id in zip(episodic_items, episodic_sqlite_ids):
    row = episodic_sqlite_readback_by_id[sqlite_id]
    assert row["memory_item_type"] == item.get("memory_item_type", "learning_event")
    assert set(item.get("source_message_ids", [])).issubset(json.loads(row["source_message_ids"]))
    assert set(item.get("source_turn_indices", [])).issubset(json.loads(row["source_turn_indices"]))
assert len(generated_episodic_insert_verification_rows) == len(episodic_items)

display({"sqlite_db_path": str(SQLITE_DB_PATH.relative_to(PROJECT_ROOT)), "sqlite_db_exists": SQLITE_DB_PATH.exists(), "ltm_ids": ltm_sqlite_ids, "episodic_ids": episodic_sqlite_ids, "counts": sqlite_counts, "ltm_inserted_rows": ltm_insert_verification_rows, "ltm_sqlite_readback_rows": ltm_sqlite_readback_rows, "episodic_sqlite_readback_rows": episodic_sqlite_readback_rows, "episodic_inserted_rows": generated_episodic_insert_verification_rows})


In [ ]:
import hashlib
import math

from episodic_schema import get_episodic_chroma_collection
from memory.ltm import ensure_ltm_vector_collection

display(Markdown("### Chroma 영속화"))

def compact_embedding(text: str, dims: int = 32) -> list[float]:
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    raw = [digest[i % len(digest)] / 255 for i in range(dims)]
    norm = math.sqrt(sum(value * value for value in raw)) or 1
    return [value / norm for value in raw]

ltm_collection = ensure_ltm_vector_collection(chroma_path=CHROMA_STORE_PATH)
episodic_collection = get_episodic_chroma_collection(CHROMA_STORE_PATH)

for collection in [ltm_collection, episodic_collection]:
    existing_ids = collection.get().get("ids", []) if collection is not None else []
    if existing_ids:
        collection.delete(ids=existing_ids)

ltm_chroma_ids = [item.get("id") or f"gdg-ltm-{index}" for index, item in enumerate(ltm_items)]
expected_ltm_metadata_by_id = {
    chroma_id: {
        "session_id": item["session_id"],
        "summary": item["summary"],
        "struggles": json.dumps(item.get("struggles", []), ensure_ascii=False),
        "strengths": json.dumps(item.get("strengths", []), ensure_ascii=False),
        "confusions": json.dumps(item.get("confusions", []), ensure_ascii=False),
        "topic_tags": json.dumps(item.get("topic_tags", []), ensure_ascii=False),
    }
    for chroma_id, item in zip(ltm_chroma_ids, ltm_items)
}
if ltm_items:
    ltm_collection.upsert(
        ids=ltm_chroma_ids,
        documents=[item["summary"] for item in ltm_items],
        embeddings=[compact_embedding(item["summary"]) for item in ltm_items],
        metadatas=[expected_ltm_metadata_by_id[chroma_id] for chroma_id in ltm_chroma_ids],
    )

ltm_chroma_readback = ltm_collection.get(
    ids=ltm_chroma_ids,
    include=["documents", "embeddings", "metadatas"],
) if ltm_items else {"ids": [], "documents": [], "embeddings": [], "metadatas": []}
assert len(ltm_chroma_readback["ids"]) == len(ltm_items)
assert ltm_chroma_readback["documents"] == [item["summary"] for item in ltm_items]
assert len(ltm_chroma_readback["embeddings"]) == len(ltm_items)
assert all(len(embedding) == 32 for embedding in ltm_chroma_readback["embeddings"])
ltm_chroma_metadata_by_id = dict(zip(ltm_chroma_readback["ids"], ltm_chroma_readback["metadatas"]))
assert ltm_chroma_metadata_by_id == expected_ltm_metadata_by_id

episodic_chroma_ids = [item.get("episodic_id") or f"gdg-episodic-{index}" for index, item in enumerate(episodic_items)]
episodic_documents = [item["topic"] for item in episodic_items]
expected_episodic_metadata_by_id = {
    chroma_id: {
        "episodic_id": chroma_id,
        "topic": item["topic"],
        "topic_tags": json.dumps(item.get("topic_tags", []), ensure_ascii=False),
        "strengths": json.dumps(item.get("strengths", []), ensure_ascii=False),
        "weaknesses": json.dumps(item.get("weaknesses", []), ensure_ascii=False),
        "questions": json.dumps(item.get("questions", []), ensure_ascii=False),
        "memory_item_type": item.get("memory_item_type", "learning_event"),
    }
    for chroma_id, item in zip(episodic_chroma_ids, episodic_items)
}
if episodic_items and episodic_collection is not None:
    episodic_collection.upsert(
        ids=episodic_chroma_ids,
        documents=episodic_documents,
        embeddings=[compact_embedding(" ".join([item["topic"], *item.get("questions", [])])) for item in episodic_items],
        metadatas=[expected_episodic_metadata_by_id[chroma_id] for chroma_id in episodic_chroma_ids],
    )

episodic_chroma_readback = episodic_collection.get(
    ids=episodic_chroma_ids,
    include=["documents", "embeddings", "metadatas"],
) if episodic_items and episodic_collection is not None else {"ids": [], "documents": [], "embeddings": [], "metadatas": []}
assert len(episodic_chroma_readback["ids"]) == len(episodic_items)
assert episodic_chroma_readback["documents"] == episodic_documents
assert len(episodic_chroma_readback["embeddings"]) == len(episodic_items)
assert all(len(embedding) == 32 for embedding in episodic_chroma_readback["embeddings"])
episodic_chroma_metadata_by_id = dict(zip(episodic_chroma_readback["ids"], episodic_chroma_readback["metadatas"]))
assert episodic_chroma_metadata_by_id == expected_episodic_metadata_by_id
ltm_chroma_count = ltm_collection.count()
episodic_chroma_count = episodic_collection.count() if episodic_collection else 0
assert ltm_chroma_count == len(ltm_items)
assert episodic_chroma_count == len(episodic_items)
chroma_validation_summary = {
    "ltm_expected_count": len(ltm_items),
    "ltm_actual_count": ltm_chroma_count,
    "episodic_expected_count": len(episodic_items),
    "episodic_actual_count": episodic_chroma_count,
    "ltm_metadata": ltm_chroma_metadata_by_id,
    "episodic_metadata": episodic_chroma_metadata_by_id,
}

display(chroma_validation_summary)
display({"chroma_store_path": CHROMA_STORE_PATH, "chroma_sqlite_exists": (CHROMA_STORE_PATH / "chroma.sqlite3").exists(), "ltm_count": ltm_chroma_count, "episodic_count": episodic_chroma_count, "ltm_documents": ltm_chroma_readback["documents"], "ltm_embedding_dimensions": [len(embedding) for embedding in ltm_chroma_readback["embeddings"]], "episodic_documents": episodic_chroma_readback["documents"], "episodic_embedding_dimensions": [len(embedding) for embedding in episodic_chroma_readback["embeddings"]]})


## 7. 검색 검증

이 단계에서는 SQLite 조회와 Chroma 유사도 검색을 실행해 저장된 메모리가 다시 검색되는지 확인합니다. 검색 결과에는 저장된 문서, 메타데이터, 점검용 쿼리 결과를 표시해 사용자가 영속화가 실제로 동작했는지 검증할 수 있게 합니다.

- 요약: 저장된 메모리를 다시 조회하고 검색 결과를 화면에 표시합니다.


In [ ]:
display(Markdown("### SQLite 검색 검증"))
assert SQLITE_DB_PATH.name == "gemini_handson.db"
assert SQLITE_DB_PATH.parent.name == "data"
with sqlite3.connect(SQLITE_DB_PATH) as conn:
    sqlite_ltm_rows = conn.execute(
        "SELECT id, session_id, summary, topic_tags FROM ltm ORDER BY id DESC LIMIT 3"
    ).fetchall()
    sqlite_episodic_rows = conn.execute(
        "SELECT episodic_id, topic, topic_tags, memory_item_type FROM episodic_memory ORDER BY last_updated DESC LIMIT 3"
    ).fetchall()
    generated_ltm_readback_rows = conn.execute(
        f"""
        SELECT generated_ltm_id, ltm_sqlite_id, session_id, summary
        FROM generated_ltm_memory
        WHERE generated_ltm_id IN ({','.join('?' for _ in ltm_items)})
        ORDER BY generated_ltm_id
        """,
        [item["id"] for item in ltm_items],
    ).fetchall() if ltm_items else []
    generated_episodic_readback_rows = conn.execute(
        f"""
        SELECT generated_episodic_id, episodic_sqlite_id, topic, memory_item_type
        FROM generated_episodic_memory
        WHERE generated_episodic_id IN ({','.join('?' for _ in episodic_items)})
        ORDER BY generated_episodic_id
        """,
        [item["episodic_id"] for item in episodic_items],
    ).fetchall() if episodic_items else []

assert len(generated_ltm_readback_rows) == len(ltm_items)
assert len(generated_episodic_readback_rows) == len(episodic_items)
sqlite_readback_summary = {
    "sqlite_db_path": str(SQLITE_DB_PATH.relative_to(PROJECT_ROOT)),
    "generated_ltm_readback_rows": generated_ltm_readback_rows,
    "generated_episodic_readback_rows": generated_episodic_readback_rows,
}
display(sqlite_readback_summary)

query_text = (
    (ltm_items[0]["summary"] if ltm_items else "")
    or (episodic_items[0]["topic"] if episodic_items else "학습 메모리")
)
query_embedding = compact_embedding(query_text)

display(Markdown("### Chroma 검색 검증"))
ltm_search_results = ltm_collection.query(
    query_embeddings=[query_embedding],
    n_results=min(3, max(1, ltm_collection.count())),
) if ltm_collection.count() else {"documents": [[]], "metadatas": [[]]}
episodic_search_results = episodic_collection.query(
    query_embeddings=[query_embedding],
    n_results=min(3, max(1, episodic_collection.count())),
) if episodic_collection and episodic_collection.count() else {"documents": [[]], "metadatas": [[]]}

ltm_search_hit_count = len(ltm_search_results.get("documents", [[]])[0])
episodic_search_hit_count = len(episodic_search_results.get("documents", [[]])[0])
assert ltm_search_hit_count >= min(1, len(ltm_items))
assert episodic_search_hit_count >= min(1, len(episodic_items))
search_verification_summary = {
    "query_text": query_text,
    "ltm_search_hit_count": ltm_search_hit_count,
    "episodic_search_hit_count": episodic_search_hit_count,
    "ltm_search_results": ltm_search_results,
    "episodic_search_results": episodic_search_results,
}
display(search_verification_summary)

retrieval_context = {
    "query_text": query_text,
    "sqlite": {
        "ltm_rows": sqlite_ltm_rows,
        "episodic_rows": sqlite_episodic_rows,
        "generated_ltm_readback_rows": generated_ltm_readback_rows,
        "generated_episodic_readback_rows": generated_episodic_readback_rows,
    },
    "chroma": {
        "ltm": ltm_search_results,
        "episodic": episodic_search_results,
    },
}
display(retrieval_context)


## 8. Gemini 챗봇 응답 생성

이 단계에서는 검색된 메모리 맥락을 바탕으로 `gemini-2.5-flash`를 호출해 챗봇 응답을 생성합니다. 최종 출력은 메모리 승격, 저장, 검색이 실제 대화 응답 생성까지 이어지는지 확인하는 사용자 검증 지점입니다.

- 요약: 검색된 메모리를 활용해 Gemini 챗봇 응답을 생성하고 표시합니다.


In [ ]:
chatbot_question = "저번에 내가 어려워했던 내용 다시 설명해줘"
chatbot_query_embedding = compact_embedding(chatbot_question)

chatbot_ltm_search_results = ltm_collection.query(
    query_embeddings=[chatbot_query_embedding],
    n_results=min(3, max(1, ltm_collection.count())),
    include=["documents", "metadatas", "distances"],
) if ltm_collection.count() else {"ids": [[]], "documents": [[]], "metadatas": [[]], "distances": [[]]}
chatbot_episodic_search_results = episodic_collection.query(
    query_embeddings=[chatbot_query_embedding],
    n_results=min(3, max(1, episodic_collection.count())),
    include=["documents", "metadatas", "distances"],
) if episodic_collection and episodic_collection.count() else {"ids": [[]], "documents": [[]], "metadatas": [[]], "distances": [[]]}

def parse_metadata_json_list(metadata, key):
    value = (metadata or {}).get(key)
    if value in (None, ""):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, list) else [parsed]
        except json.JSONDecodeError:
            return [part.strip() for part in value.split(",") if part.strip()]
    return [value]

def chroma_hits(results):
    ids = results.get("ids", [[]])[0]
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]
    return [
        {"id": item_id, "document": document, "metadata": metadata or {}, "distance": distance}
        for item_id, document, metadata, distance in zip(ids, documents, metadatas, distances)
    ]

def format_ltm_hit_for_chatbot(hit):
    metadata = hit["metadata"]
    return {
        "memory_type": "LTM",
        "id": hit["id"],
        "distance": hit["distance"],
        "session_id": metadata.get("session_id"),
        "summary": metadata.get("summary") or hit["document"],
        "struggles": parse_metadata_json_list(metadata, "struggles"),
        "strengths": parse_metadata_json_list(metadata, "strengths"),
        "confusions": parse_metadata_json_list(metadata, "confusions"),
        "topic_tags": parse_metadata_json_list(metadata, "topic_tags"),
    }

def format_episodic_hit_for_chatbot(hit):
    metadata = hit["metadata"]
    return {
        "memory_type": "Episodic",
        "id": metadata.get("episodic_id") or hit["id"],
        "distance": hit["distance"],
        "topic": metadata.get("topic") or hit["document"],
        "strengths": parse_metadata_json_list(metadata, "strengths"),
        "weaknesses": parse_metadata_json_list(metadata, "weaknesses"),
        "questions": parse_metadata_json_list(metadata, "questions"),
        "topic_tags": parse_metadata_json_list(metadata, "topic_tags"),
    }

chatbot_retrieval_context = {
    "query": chatbot_question,
    "ltm_context": [format_ltm_hit_for_chatbot(hit) for hit in chroma_hits(chatbot_ltm_search_results)],
    "episodic_context": [format_episodic_hit_for_chatbot(hit) for hit in chroma_hits(chatbot_episodic_search_results)],
}

display(Markdown("### 사용자 질문 기반 Semantic Search 정리 Context"))
display(chatbot_retrieval_context)

chatbot_prompt = f"""
당신은 학습 메모리를 활용하는 한국어 튜터 챗봇입니다.
사용자 질문으로 semantic search한 LTM과 에피소드 메모리만 근거로 답하세요.
학습자의 강점, 어려움, 이전 질문 패턴을 반영해 사용자 답변에 대답하세요.

Semantic search 메모리:
{json.dumps(chatbot_retrieval_context, ensure_ascii=False, indent=2)}

사용자 질문: {chatbot_question}
"""

chatbot_response = client.models.generate_content(model=CHATBOT_MODEL, contents=chatbot_prompt)
chatbot_reply = (getattr(chatbot_response, "text", None) or "").strip()
assert chatbot_reply, "Gemini 챗봇 응답이 비어 있습니다."

display(Markdown("### Gemini 챗봇 응답"))
display({"model": CHATBOT_MODEL, "question": chatbot_question, "retrieval_context": chatbot_retrieval_context, "response": chatbot_reply})
